# EmPath — connect and review the walking network
**Purpose:** find candidate origin/service connections, identify unreachable pairs and calculate distance-based baselines. This does not fit GeoAI weights or certify safe/legal walking. Keep the full graph so cooler alternative routes remain available.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists(): ROOT = ROOT.parent
assert (ROOT / 'scripts').exists(), 'Open this notebook from the EmPath project.'
sys.path.insert(0, str(ROOT / 'scripts'))

## 1. Generate the first review list
The first run chooses one address-containing block per ZIP and mapped food/pharmacy POIs. These are examples, not verified residential origins or entrances. Temples may be missing from the source inventory. Nearest-node gaps are reported, not added as straight-line walkable links.

In [2]:
from connect_pilot_network import run
report = run()
report

{'nodes': 31372,
 'directed_edges': 93452,
 'weak_components': 179,
 'locations': 89,
 'od_pairs': 340,
 'unreachable_pairs': 12,
 'note': 'Provisional nearest-node screening, not verified entrances. Network-only times assume 1.2 m/s, exclude connector gaps and crossing waits. All alternatives retained. Review connections before accessibility claims.'}

In [3]:
import pandas as pd
OUT = ROOT / 'output/network_connections'
connections = pd.read_csv(OUT / 'candidate_connections.csv')
display(connections)
pairs = pd.read_csv(OUT / 'baseline_od_review.csv')
display(pairs.head(20))
display(pairs[pairs.network_reachable.eq(False)])

,location_id,role,label,entrance_verified,category,network_node,connector_distance_m,connection_status
0,block_480850316283000,origin,"6000 CUSTER RD, PLANO, 75023",False,NaN,13011072784,18.028379,review_nearby_node
1,block_480850316572008,origin,"8050 DOMINION PKWY, PLANO, 75024",False,NaN,7273418380,23.143064,review_nearby_node
2,block_480850316791001,origin,"3225 RUSSELL CREEK DR, PLANO, 75025",False,NaN,13667186853,125.121241,review_large_gap
3,block_480850320082010,origin,"2916 CHARTER OAK DR, PLANO, 75074",False,NaN,150374477,51.126437,review_nearby_node
4,poi_1,destination,India Bazaar,False,supermarket,5548383500,22.815949,review_nearby_node
...,...,...,...,...,...,...,...,...
84,poi_224,destination,Sprouts Farmers Market,False,supermarket,10308921114,41.476589,review_nearby_node
85,poi_225,destination,Walmart Neighborhood Market,False,supermarket,5797694599,44.471654,review_nearby_node
86,poi_228,destination,7-Eleven,False,convenience,5548383865,33.792967,review_nearby_node
87,poi_235,destination,7-Eleven,False,convenience,8427285430,20.367843,review_nearby_node


,origin,destination,network_reachable,network_distance_m,baseline_network_minutes,origin_gap_m,destination_gap_m,access_validated
0,block_480850316283000,poi_1,True,2891.992380,40.166561,18.028379,22.815949,False
1,block_480850316283000,poi_2,True,4153.009239,57.680684,18.028379,42.342591,False
2,block_480850316283000,poi_3,True,5383.490409,74.770700,18.028379,38.130298,False
3,block_480850316283000,poi_4,True,5429.407422,75.408436,18.028379,19.909796,False
4,block_480850316283000,poi_5,True,384.612875,5.341845,18.028379,20.621767,False
5,block_480850316283000,poi_6,True,10355.582970,143.827541,18.028379,53.251048,False
6,block_480850316283000,poi_7,True,3898.731744,54.149052,18.028379,22.638552,False
7,block_480850316283000,poi_10,True,3312.250790,46.003483,18.028379,12.673946,False
8,block_480850316283000,poi_11,True,5926.417655,82.311356,18.028379,19.382353,False
9,block_480850316283000,poi_14,True,5042.532465,70.035173,18.028379,32.382997,False


,origin,destination,network_reachable,network_distance_m,baseline_network_minutes,origin_gap_m,destination_gap_m,access_validated
18,block_480850316283000,poi_24,False,NaN,NaN,18.028379,24.823969,False
19,block_480850316283000,poi_25,False,NaN,NaN,18.028379,43.700307,False
21,block_480850316283000,poi_28,False,NaN,NaN,18.028379,36.853497,False
103,block_480850316572008,poi_24,False,NaN,NaN,23.143064,24.823969,False
104,block_480850316572008,poi_25,False,NaN,NaN,23.143064,43.700307,False
106,block_480850316572008,poi_28,False,NaN,NaN,23.143064,36.853497,False
188,block_480850316791001,poi_24,False,NaN,NaN,125.121241,24.823969,False
189,block_480850316791001,poi_25,False,NaN,NaN,125.121241,43.700307,False
191,block_480850316791001,poi_28,False,NaN,NaN,125.121241,36.853497,False
273,block_480850320082010,poi_24,False,NaN,NaN,51.126437,24.823969,False


## 2. Review entrances before using results
- Open `output/network_connections/candidate_locations.csv`. Save a copy as `reviewed_locations.csv` in the same folder.
- Keep a small set of origins and relevant destinations. Add missing temples or other services.
- Move longitude/latitude to actual pedestrian access points using imagery and municipal paths. Confirm access, barriers, sidewalk links and crossings. Record your evidence separately; mark `entrance_verified` only after review.
- Rerun this notebook. A nearby node can still be on the wrong side of a divided road. Large gaps require inspection; do not increase a snapping threshold simply to make everything connect.
- `access_validated` remains false intentionally: entrance review alone does not validate the complete route.

## 3. Handoff for routing
Outputs: candidate connection GeoPackage/CSV, OD baseline CSV, structural audit JSON. Times use 1.2 m/s and exclude connector gaps/crossing waits. Inspect example routes before final use. Keep origin-to-origin *recommendations* out of the results; keep residential streets in the underlying graph. Next, use documented nonnegative friction costs and compare lowest-friction routes against these distance baselines.